# Common Libraries

In [1]:
import os, shutil
if shutil.which('nvidia-smi') is not None:
    os.environ['MUJOCO_GL'] = 'egl'
import mujoco, mink
from loop_rate_limiters import RateLimiter
import numpy as np
import ipywidgets as widgets
from mink import Configuration, SE3, SO3
from IPython.display import display, clear_output
import mediapy as media

# Params

In [2]:
myosim_dir_path = "/home/seojin/myosuite/myosuite/simhive/myo_sim/arm"
model_name = "myoarm.xml"

solver = "quadprog"
pos_threshold = 1e-4
ori_threshold = 1e-4
max_iters = 20

fps = 60
dt = 1 / fps

In [3]:
os.chdir(myosim_dir_path)

# Model specification

In [4]:
xml_string = f"""
        <mujoco model="MyoArm with Mocap">
            <include file="{model_name}"/>
            <worldbody>
                <body name="target" pos="0 0 0" quat="0 1 0 0" mocap="true">
                    <geom type="box" size=".15 .15 .15" contype="0" conaffinity="0" rgba=".6 .3 .3 .2"/>
                </body>
            </worldbody>
        </mujoco>
        """

# Initialize model & data

In [5]:
model = mujoco.MjModel.from_xml_string(xml_string)
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)

# Initialize the mocap target at the end-effector site.
mink.move_mocap_to_frame(model, data, "target", "S_grasp", "site")

# IK Configuration

In [17]:
configuration = mink.Configuration(model)
configuration.update(data.qpos)
data.qpos[:] = configuration.q
mujoco.mj_step(model, data)

renderer = mujoco.Renderer(configuration.model, height=400, width=600)

end_effector_task = mink.FrameTask(frame_name="S_grasp",
                                   frame_type="site",
                                   position_cost=1.0,
                                   orientation_cost=1.0,
                                   lm_damping=1.0)
posture_task =  mink.PostureTask(model=model, cost=1e-2)
tasks = [
    end_effector_task,
    posture_task,
]
posture_task.set_target_from_configuration(configuration)

camera = mujoco.MjvCamera()
camera.distance = 3.0

In [30]:
qpos_history = []

def update_target(x, y, z,
                  w_o, x_o, y_o, z_o):
    # Set new target
    new_translation = np.array([x, y, z])
    orientation = np.array([w_o, x_o, y_o, z_o])
    target_val = np.concatenate([orientation, new_translation])
    new_target = SE3(wxyz_xyz = target_val)

    # Set target position and orientation
    end_effector_task.set_target(new_target)
    configuration.data.mocap_pos[0] = new_target.wxyz_xyz[4:]
    configuration.data.mocap_quat[0] = new_target.wxyz_xyz[:4]
    
    # Rendering
    formatted_text = ", ".join([f"{val:.2f}" for val in target_val])
    data_label.value = f"wxyz_xyz: [{formatted_text}]"

    mujoco.mj_forward(configuration.model, configuration.data)
    renderer.update_scene(configuration.data, camera = camera)
    pixels = renderer.render()
    
    with output_area:
        clear_output(wait=True)
        media.show_image(pixels)

def run_simulation(b):
    # Perform inverse kinematics
    vel = solve_ik(configuration, [tasks], dt, solver = solver_name)
    configuration.integrate_inplace(vel, dt)
    
    # Update the physical state of the simulation
    mujoco.mj_forward(configuration.model, configuration.data)
    qpos_history.append(configuration.data.qpos.copy())
    
    # Save the current frame for video
    renderer.update_scene(configuration.data, camera = camera)
    pixels = renderer.render()

    with output_area:
        clear_output(wait=True)
        media.show_image(pixels)

In [31]:
# Make widgets
data_label = widgets.Label(value="wxyz_xyz: [0, 0, 0, 0, 0, 0, 0]")
x_slider = widgets.FloatSlider(value=0.2, min=-2.0, max=2.0, step=0.01, description='Target X:')
y_slider = widgets.FloatSlider(value=0.3, min=-2.0, max=2.0, step=0.01, description='Target Y:')
z_slider = widgets.FloatSlider(value=0.3, min=-2.0, max=2.0, step=0.01, description='Target Z:')

w_o_slider = widgets.FloatSlider(value=0, min=-1.0, max=1.0, step=0.01, description='Target w_o:')
x_o_slider = widgets.FloatSlider(value=0, min=-1.0, max=1.0, step=0.01, description='Target x_o:')
y_o_slider = widgets.FloatSlider(value=0, min=-1.0, max=1.0, step=0.01, description='Target y_o:')
z_o_slider = widgets.FloatSlider(value=0, min=-1.0, max=1.0, step=0.01, description='Target z_o:')

sim_button = widgets.Button(
    description='Run Simulation',
    button_style='success', # 초록색 버튼
    icon='play'
)

output_area = widgets.Output()

# Layout and event connection
ui = widgets.VBox([
    widgets.HBox([data_label, w_o_slider]),
    widgets.HBox([x_slider, x_o_slider]),
    widgets.HBox([y_slider, y_o_slider]),
    widgets.HBox([z_slider, z_o_slider]),
    sim_button,
])

out = widgets.interactive_output(update_target, {
    'x': x_slider, 'y': y_slider, 'z': z_slider,
    'w_o' : w_o_slider, 'x_o': x_o_slider, 'y_o': y_o_slider, 'z_o': z_o_slider,
})

sim_button.on_click(run_simulation)

display(ui, output_area)
update_target(x_slider.value, y_slider.value, z_slider.value, 
             w_o_slider.value, x_o_slider.value, y_o_slider.value, z_o_slider.value)


Output()

In [28]:
qpos_history.append(None)

In [32]:
qpos_history

[]